Nesta etapa criamos novas variáveis a partir dos dados originais para melhorar a capacidade dos modelos de machine learning em identificar transações fraudulentas.

As features criadas buscam capturar padrões de comportamento suspeito, como transações de alto valor, horários incomuns e características de risco associadas às transações.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, log, col, avg, stddev, count
from pyspark.ml.feature import StringIndexer 	
from pyspark.sql.functions import abs as spark_abs

caminho = "/Workspace/Users/henriquegmour4@gmail.com/Fraud Detection in Financial Transactions using PySpark/fraud-detection-spark/data/raw/credit_card_fraud.csv"

fraud_df = spark.read.csv(caminho, header=True, inferSchema=True)

fraud_df.createOrReplaceTempView("fraud_transactions")

**1. Criando features temporais (hour, day_of_week, is_weekend)**

In [0]:
# Transação com alto valor
fraud_df = fraud_df.withColumn("high_value_transaction", 
    when(col("amount") > 1000, 1).otherwise(0))

In [0]:
# Transação internacional com valor alto
fraud_df = fraud_df.withColumn("foreign_high_risk", 
    when((fraud_df.foreign_transaction == 1) & (fraud_df.amount > 500), 1).otherwise(0))


In [0]:
# Transação Noturna com alto valor
fraud_df = fraud_df.withColumn("night_transaction", 
    when((fraud_df.transaction_hour >= 0) & (fraud_df.transaction_hour <= 6), 1).otherwise(0))

In [0]:
# Registro de quantia 
fraud_df = fraud_df.withColumn("amount_log",log(fraud_df.amount + 1))


In [0]:
# Combinar múltiplos fatores de risco
fraud_df = fraud_df.withColumn("foreign_high_risk", 
    when((fraud_df.foreign_transaction == 1) & (fraud_df.night_transaction == 1), 1).otherwise(0))

In [0]:
print(fraud_df.columns)

In [0]:
# Verificação dos valores criados
print("Novas features:")
fraud_df.select("high_value_transaction", "night_transaction", 
                "amount_log", "foreign_high_risk").describe().show()

# Verificar correlação com fraude
for col_name in ["high_value_transaction", "night_transaction", "foreign_high_risk"]:
    corr = fraud_df.stat.corr(col_name, "is_fraud")
    print(f"Correlação {col_name} com is_fraud: {corr:.4f}")

# Verificar distribuição de amount_log
fraud_df.select("amount", "amount_log").show(10)

**2. Criando features de agregação (merchant_avg_amount, category_avg_amount)**

In [0]:
# Índice de categoria de comerciante
indexer = StringIndexer(inputCol = "merchant_category", outputCol="merchant_category_index")
fraud_df = indexer.fit(fraud_df).transform(fraud_df)

In [0]:
# Média de amount por merchant_category
category_avg = fraud_df.groupBy("merchant_category").agg(
    avg("amount").alias("category_avg_amount")
)

fraud_df = fraud_df.join(category_avg, "merchant_category", "left")

print("Feature category_avg_amount criada")
fraud_df.select("merchant_category", "amount", "category_avg_amount").show(10)

In [0]:
# Criar feature de desvio em relação à média da categoria

fraud_df = fraud_df.withColumn("amount_deviation_category",
    col("amount") - col("category_avg_amount")
)

print("Feature amount_deviation_category criada")
fraud_df.select("amount", "category_avg_amount", "amount_deviation_category").show(10)

In [0]:
# Features de agregação adicionais (opcional mas recomendado)

from pyspark.sql.functions import stddev, count

# Desvio padrão e contagem por categoria
category_stats = fraud_df.groupBy("merchant_category").agg(
    stddev("amount").alias("category_std_amount"),
    count("*").alias("category_transaction_count")
)

fraud_df = fraud_df.join(category_stats, "merchant_category", "left")

print("Features adicionais criadas")
fraud_df.select("merchant_category", "category_std_amount", "category_transaction_count").show(10)

In [0]:
# Z-score por categoria (normalização)

fraud_df = fraud_df.withColumn("amount_zscore_category",
    (col("amount") - col("category_avg_amount")) / col("category_std_amount")
)

# Tratar divisão por zero (quando std = 0)
fraud_df = fraud_df.fillna({"amount_zscore_category": 0})

print("Feature amount_zscore_category criada")
fraud_df.select("amount", "category_avg_amount", "category_std_amount", "amount_zscore_category").show(10)

In [0]:
# Validação das features de agregação

print("Estatísticas das features de agregação:")
fraud_df.select("category_avg_amount", "amount_deviation_category", 
                "category_std_amount", "amount_zscore_category").describe().show()

# Verificar correlação com fraude
print("\nCorrelação com is_fraud:")
agg_features = ["category_avg_amount", "amount_deviation_category", 
                "category_std_amount", "amount_zscore_category", 
                "category_transaction_count"]

for col_name in agg_features:
    if col_name in fraud_df.columns:
        corr = fraud_df.stat.corr(col_name, "is_fraud")
        print(f"{col_name}: {corr:.4f}")

# Verificar valores nulos
from pyspark.sql.functions import sum as spark_sum
print("\nValores nulos:")
fraud_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(f"{c}_nulls")
    for c in agg_features if c in fraud_df.columns
]).show()

In [0]:
print(fraud_df.columns)

**3. Criando features de desvio (amount_deviation_merchant, amount_deviation_category)**


In [0]:
# Criando features de agregação de desvio por categoria e tipo de transação
category_foreign_avg = fraud_df.groupBy("merchant_category", "foreign_transaction").agg(
    avg("amount").alias("category_foreign_avg_amount")
)

fraud_df = fraud_df.join(category_foreign_avg, 
                         ["merchant_category", "foreign_transaction"], "left")

fraud_df = fraud_df.withColumn("amount_deviation_category_foreign",
    col("amount") - col("category_foreign_avg_amount")
)

print("Feature amount_deviation_category_foreign criada")
fraud_df.select("merchant_category", "foreign_transaction", "amount", 
                "category_foreign_avg_amount", "amount_deviation_category_foreign").show(10)

In [0]:
# Desvio por categoria + hora do dia
category_hour_avg = fraud_df.groupBy("merchant_category", "transaction_hour").agg(
    avg("amount").alias("category_hour_avg_amount")
)

fraud_df = fraud_df.join(category_hour_avg, 
                        ["merchant_category", "transaction_hour"], "left")

fraud_df = fraud_df.withColumn("amount_deviation_category_hour",
    col("amount") - col("category_hour_avg_amount")
)

print("Feature amount_deviation_category_hour criada")
fraud_df.select("merchant_category", "transaction_hour", "amount", 
                "category_hour_avg_amount", "amount_deviation_category_hour").show(10)

In [0]:
# Desvio por categoria + device_trust_score
category_device_avg = fraud_df.groupBy("merchant_category", "device_trust_score").agg(
    avg("amount").alias("category_device_avg_amount")
)

fraud_df = fraud_df.join(category_device_avg, 
                        ["merchant_category", "device_trust_score"], "left")

fraud_df = fraud_df.withColumn("amount_deviation_category_device",
    col("amount") - col("category_device_avg_amount")
)

print("Feature amount_deviation_category_device criada")
fraud_df.select("merchant_category", "device_trust_score", "amount", 
                "category_device_avg_amount", "amount_deviation_category_device").show(10)

In [0]:
# Verificando desvios
deviation_features = [
    "amount_deviation_category",
    "amount_deviation_category_foreign",
    "amount_deviation_category_hour",
    "amount_deviation_category_device"
]

fraud_df.select(deviation_features).describe().show()

In [0]:
# Criando features de desvio absoluto (magnitude do desvio)
fraud_df = fraud_df.withColumn("abs_deviation_category",
    spark_abs(col("amount_deviation_category"))
)

fraud_df = fraud_df.withColumn("abs_deviation_category_foreign",
    spark_abs(col("amount_deviation_category_foreign"))
)

print("Features de desvio absoluto criadas")
fraud_df.select("amount_deviation_category", "abs_deviation_category",
                "amount_deviation_category_foreign", "abs_deviation_category_foreign").show(10)

**4. Encodando variáveis categóricas com StringIndexer**

In [0]:
# Identificar colunas categóricas do tipo string
categorical_cols = [col_name for col_name, dtype in fraud_df.dtypes if dtype == 'string']

print(f"Colunas categóricas encontradas: {categorical_cols}")

In [0]:
# Aplicar StringIndexer para merchant_category (onde não foi feito)

if "merchant_category_index" not in fraud_df.columns:
    indexer_category = StringIndexer(
        inputCol="merchant_category",
        outputCol="merchant_category_index"
    )
    fraud_df = indexer_category.fit(fraud_df).transform(fraud_df)
    print("merchant_category_index criada")
else:
    print("merchant_category_index já existe")

fraud_df.select("merchant_category", "merchant_category_index").distinct().show()

In [0]:
# Validar encoding de merchant_category

print("Validação do encoding de merchant_category:")

# Mostrar mapeamento categoria -> index
fraud_df.select("merchant_category", "merchant_category_index").distinct().orderBy("merchant_category_index").show()

# Contar valores únicos
n_categories = fraud_df.select("merchant_category").distinct().count()
n_indexes = fraud_df.select("merchant_category_index").distinct().count()

print(f"Total de categorias únicas: {n_categories}")
print(f"Total de índices únicos: {n_indexes}")
print(f"Encoding correto: {n_categories == n_indexes}")

In [0]:
# Verificar se há valores nulos
print(f"merchant_category_nulls: {fraud_df.filter(col('merchant_category').isNull()).count()}")
print(f"merchant_category_index_nulls: {fraud_df.filter(col('merchant_category_index').isNull()).count()}")

In [0]:
# Listar todas as features prontas para modelagem
numeric_features = [col_name for col_name, dtype in fraud_df.dtypes 
                   if dtype in ['int', 'integer', 'double', 'bigint'] 
                   and col_name not in ['transaction_id', 'is_fraud']]

print(f"\nTotal de features numéricas para modelagem: {len(numeric_features)}")
print("\nFeatures:")
for feat in sorted(numeric_features):
    print(f"  - {feat}")

In [0]:
fraud_df.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fraud_features")